In [5]:
current_step = 'step_009'

In [6]:
!apt install swig cmake ffmpeg xvfb python3-opengl
!pip install pyvirtualdisplay imageio[ffmpeg]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-opengl is already the newest version (3.1.5+dfsg-1).
swig is already the newest version (4.0.2-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.15).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [7]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

%env MUJOCO_GL=egl

from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

env: MUJOCO_GL=egl


In [24]:
# Prepare to load data from google drive
from google.colab import drive
import datetime

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
workDir = os.path.join(gdrive_path, 'My Drive', 'Research', current_step)
print('WorkDir:', workDir)

log_dir = os.path.join(workDir, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
print('LogDir:', log_dir)

# create folder if it doesn't exists
if not os.path.exists(log_dir):
  os.makedirs(log_dir)

tf_log_dir = os.path.join(log_dir, 'tensorboard_logs')
print('TfLogDir:', tf_log_dir)

# create folder if it doesn't exists
if not os.path.exists(tf_log_dir):
  os.makedirs(tf_log_dir)

# best models
to_load_path = os.path.join(workDir, 'best')
algos = ["a2c", "ddpg", "ppo", "sac", "td3"]
for algo in algos:
  algo_load_path = os.path.join(to_load_path, algo)
  if not os.path.exists(algo_load_path):
    os.makedirs(algo_load_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WorkDir: /content/drive/My Drive/Research/step_009
LogDir: /content/drive/My Drive/Research/step_009/20250905-195633
TfLogDir: /content/drive/My Drive/Research/step_009/20250905-195633/tensorboard_logs


In [10]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

Error: /content/rl-zoo : No such file or directory
Error: /content/gym_darwin_op3 : No such file or directory
Error: /content/videos : No such file or directory


In [18]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


The directory '/content/gym_darwin_op3' exists - git pull
/content/gym_darwin_op3
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 8), reused 8 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (17/17), 26.40 KiB | 659.00 KiB/s, done.
From https://github.com/Gianzanti/robofei_mestrado
   6123ddf..d88d24b  step_009   -> origin/step_009
Updating 6123ddf..d88d24b
Fast-forward
 notebooks/Research_Training.ipynb | 36 +++++++++++++++++++-----------------
 pyproject.toml                    |  2 +-
 src/robofei/env/darwin_op3.py     | 14 +++++++-------
 3 files changed, 27 insertions(+), 25 deletions(-)
/


In [19]:
!pip install -e {model_path}

Obtaining file:///content/gym_darwin_op3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for robofei (pyproject.toml) ... done
  Created wheel for robofei: filename=robofei-0.1.21-py3-none-any.whl size=1513 sha256=a5dafdffe11e2a361262ce469d1e2519134e4f8b0fa172201b7db50875f3a2fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-wsknx3k_/wheels/43/c3/48/682f53e738aa575222935107724428d2ce2d742b055ebb4adc
Successfully built robofei
  Attempting uninstall: robofei
    Found existing installation: robofei 0.1.20
    Uninstalling robofei-0.1.20:
      Successfully uninstalled robofei-0.1.20


In [13]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


The directory '/content/rl-zoo' does not exist - git clone
Cloning into '/content/rl-zoo'...
remote: Enumerating objects: 3661, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 3661 (delta 37), reused 27 (delta 16), pack-reused 3597 (from 2)
Receiving objects: 100% (3661/3661), 8.53 MiB | 17.85 MiB/s, done.
Resolving deltas: 100% (2234/2234), done.


In [14]:
!pip install -e {trainner_path}

Obtaining file:///content/rl-zoo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 8.8 MB/s eta 0:00:00
  Building editable for rl_zoo3 (pyproject.toml) ... done
  Created wheel for rl_zoo3: filename=rl_zoo3-2.7.0-0.editable-py3-none-any.whl size=4428 sha256=b13bc41f702920a9d350db9e3154184af388d0a98734dd8565a92daedb805566
  Stored in directory: /tmp/pip-ephem-wheel-cache-ylm9l6pn/wheels/2d/0b/3c/be4c09b7d9d2a891b5d1bc1e086a8fe4f4b4f016a0613b625c
Successfully built rl_zoo3


In [25]:
%cd {trainner_path}

algos = {
  "a2c": {
    "lr": 7e-4, "dev": "cpu"
  },
  'ddpg': {
    'lr': 1e-3, 'dev': 'cuda'
  },
  'ppo': {
    'lr': 3e-4, 'dev': 'cpu'
  },
  'sac': {
    'lr': 3e-4, 'dev': 'cuda'
  },
  'td3': {
    'lr': 1e-3, 'dev': 'cuda'
  }
}

n_timestep = 10_000_000
save_freq = min(100_000, int(n_timestep / 10))
eval_freq = min(200_000, int(n_timestep / 10))
max_episode_steps = 100
wrapper = [{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": max_episode_steps}}]
n_envs = 16

# weights
keep_alive_reward = 1.0
ctrl_cost_weight = 0.0001
target_distance = 2.5
forward_velocity_weight = 2.0
reach_target_reward = 20.0

for algo, value in algos.items():
  print('Training:', algo)
  config = f'research_config/{algo}.yml'
  to_load_path = os.path.join(workDir, 'best', algo, 'best_model.zip')

  train_cmd = f'python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} \
-f "{log_dir}" --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
--vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 5 \
--env-kwargs keep_alive_reward:{keep_alive_reward} ctrl_cost_weight:{ctrl_cost_weight} \
target_distance:{target_distance} forward_velocity_weight:{forward_velocity_weight} \
reach_target_reward:{reach_target_reward} --hyperparams n_envs:{n_envs} \
learning_rate:{value["lr"]} n_timesteps:{n_timestep} \nv_wrapper:"{wrapper}" \
--device {value["dev"]} -i "{to_load_path}"'

  print(train_cmd)
  !{train_cmd}

  video_cmd = f'python3 -m rl_zoo3.record_video --algo {algo} \
--env DarwinOp3-v2 -n 2000 --load-best -o "{log_dir}" -f "{log_dir}"'
  print(video_cmd)
  !{video_cmd}


/content/rl-zoo
Training: a2c
python3 train.py --algo a2c --env DarwinOp3-v2 -conf research_config/a2c.yml -f "/content/drive/My Drive/Research/step_009/20250905-195633" --tensorboard-log "/content/drive/My Drive/Research/step_009/20250905-195633/tensorboard_logs" --save-freq 100000 --vec-env subproc --eval-freq 200000 --n-eval-envs 4 --eval-episodes 12 --env-kwargs keep_alive_reward:1.0 ctrl_cost_weight:0.0001 target_distance:2.5 forward_velocity_weight:2.0 reach_target_reward:20.0 --hyperparams n_envs:16 learning_rate:0.0007 n_timesteps:10000000 
v_wrapper:"[{'gymnasium.wrappers.TimeLimit': {'max_episode_steps': 1000}}]" --device cpu -i "/content/drive/My Drive/Research/step_009/best/a2c/best_model.zip"
2025-09-05 19:57:15.728621: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757102235.749126   18479 cuda_dnn.cc:8579] Unable to register 

KeyboardInterrupt: 